<a href="https://colab.research.google.com/github/nalgo-intern/xxx/blob/test_b/hello.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install fugashi ipadic unidic-lite

In [ ]:
import pandas as pd

# データの例（0: 不満/言及なし, 1: 満足）
data = [
    {"text": "料理は絶品ですが、接客がイマイチで待ち時間も長かった。", "taste": 1, "service": 0, "congestion": 0},
    {"text": "店員さんが親切で料理も美味しく、待ち時間ゼロでした！", "taste": 1, "service": 1, "congestion": 1},
    {"text": "値段の割に味が普通。店内が混んでいて落ち着かなかった。", "taste": 0, "service": 0, "congestion": 0},
    # ※本番では数十〜数百件程度のデータを用意
]

df = pd.DataFrame(data)

In [ ]:
from transformers import AutoTokenizer
import torch

# 日本語モデルの標準「東北大BERT」を指定
MODEL_NAME = "cl-tohoku/bert-base-japanese-v3"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        inputs = self.tokenizer(
            row['text'],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        # 3つのラベルをテンソルに変換（多ラベル分類用）
        labels = torch.tensor([row['taste'], row['service'], row['congestion']], dtype=torch.float)

        return {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'labels': labels
        }

config.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/251 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/231k [00:00<?, ?B/s]

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3, # [味, 接客, 混雑] の3つ
    problem_type="multi_label_classification" # 多ラベル分類を指定
)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  447MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: cl-tohoku/bert-base-japanese-v3
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

In [ ]:
from transformers import Trainer, TrainingArguments

# Datasetの準備
dataset = ReviewDataset(df, tokenizer)

# 学習の設定
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,           # 学習回数（エポック数）
    per_device_train_batch_size=8, # バッチサイズ
    logging_steps=10,
    save_strategy="no",           # 4日間の開発なら保存は最後のみでOK
)

# Trainerの設定と学習開始
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

# 学習実行！(Colab GPUなら数十秒〜数分で終わります)
trainer.train()

# 学習済みモデルとトークナイザーを保存
model.save_pretrained("./my_aspect_bert")
tokenizer.save_pretrained("./my_aspect_bert")

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./my_aspect_bert/tokenizer_config.json',
 './my_aspect_bert/vocab.txt',
 './my_aspect_bert/added_tokens.json')

In [ ]:
def predict_aspect_score(text):
    # 1. テキストをトークナイズ
    inputs = tokenizer(text, return_tensors="pt", max_length=128, truncation=True, padding=True)

    # 🌟 【修正箇所】入力テンソルをモデルと同じデバイス (CPU or GPU) に送る
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # 2. 推論実行
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        # Sigmoid関数を通して0.0〜1.0（確率）に変換
        probs = torch.sigmoid(outputs.logits)[0].tolist()

    # 3. パーセント（%）化して返却
    results = {
        "味・品質": f"{round(probs[0] * 100)}%",
        "接客・雰囲気": f"{round(probs[1] * 100)}%",
        "混雑・待ち時間": f"{round(probs[2] * 100)}%"
    }
    return results

# 動作確認（エラーなく動くはずです！）
sample_review = "お肉は柔らかくて最高でした！ただ、接客が雑で少し並びました。"
print(predict_aspect_score(sample_review))

{'味・品質': '69%', '接客・雰囲気': '43%', '混雑・待ち時間': '30%'}


In [ ]:
# 1. ライブラリのインストール
!pip install gradio pandas plotly -q

import gradio as gr
import pandas as pd

# 2. 感情分析ロジック（ダミー関数：ここに実際のモデル/LLM処理を入れる）
def analyze_review_ui(review_text):
    if not review_text.strip():
        return "テキストを入力してください。", None, ""

    # 本来はここに機械学習/LLMの計算処理が入る
    # 例としてダミーのパーセントスコアを返す
    scores = {
        "接客・雰囲気": 0.80, # 80%
        "味・品質": 0.63,     # 63%
        "混雑・待ち時間": 0.35 # 35%
    }

    insight_text = "💡 **AIインサイト:**\n味や接客は高く評価されていますが、待ち時間に対する不満（35%）が全体の評価を押し下げている傾向があります。"

    return scores, insight_text

# 3. Gradio UIの構築
with gr.Blocks(title="口コミ要素別 感情分析ツール") as demo:
    gr.Markdown("# 🏨 口コミ要素別 満足度分析ツール")
    gr.Markdown("星評価では見えない「接客」「味」「混雑」などの個別要素を可視化します。")

    with gr.Row():
        with gr.Column():
            # 入力エリア
            input_text = gr.Textbox(
                label="分析する口コミテキスト",
                lines=5,
                placeholder="ここにGoogleマップ等の口コミを貼り付けてください..."
            )
            submit_btn = gr.Button("分析実行", variant="primary")

        with gr.Column():
            # 出力エリア
            gr.Markdown("### 【口コミ要素別 満足度スコア】")
            # gr.Label は辞書型（{"要素名": スコア(0~1)}）を渡すとバー形式で表示してくれる
            output_scores = gr.Label(label="要素別ポジティブ率")
            output_insight = gr.Markdown()

    # ボタン押下時の処理
    submit_btn.click(
        fn=analyze_review_ui,
        inputs=[input_text],
        outputs=[output_scores, output_insight]
    )

# 4. 起動（share=Trueで公開URLを発行）
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f7604f60cb41d65d9d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://f7604f60cb41d65d9d.gradio.live


In [ ]:
# 必要なライブラリのインストール
!pip install sentence-transformers transformers torch pandas mecab-python3 fugashi ipadic -q

import torch
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd

# 1. 文章の意味（ベクトル）を計算するモデル（そのまま）
embedding_model = SentenceTransformer('cl-tohoku/bert-base-japanese-whole-word-masking')

# 2. ポジ・ネガ感情判定モデルのロード（★ここを修正！）
sentiment_model_name = "koheiduck/bert-japanese-finetuned-sentiment"
sentiment_tokenizer = AutoTokenizer.from_pretrained(sentiment_model_name)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(sentiment_model_name)
sentiment_model.eval()


# ----------------------------------------------------
# 3. 事前に用意した「固定要素」の定義（辞書ではなくカテゴリ名のみ！）
# ----------------------------------------------------
FIXED_ASPECTS = {
    "宿泊施設": ["部屋の清潔さ・広さ", "接客・スタッフ対応", "温泉・お風呂・アメニティ", "朝食・食事の味"],
    "飲食店": ["料理の味・品質", "接客・サービス", "待ち時間・混雑度", "価格・コスパ"]
}

def predict_sentiment(text):
    """1文のポジティブ度(0.0〜1.0)を判定"""
    inputs = sentiment_tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        outputs = sentiment_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0]
    return probs[1].item() # ポジティブ確率

def analyze_review_with_fixed_aspects(text, genre="飲食店"):
    """手動辞書を使わず、固定要素に自動分類して集計"""
    aspect_list = FIXED_ASPECTS[genre]

    # 事前定義した固定要素をベクトル化
    aspect_embeddings = embedding_model.encode(aspect_list, convert_to_tensor=True)

    # 句点で文章を分割（1文ごとに分析）
    sentences = [s.strip() for s in text.replace("！", "。").replace("!", "。").split("。") if len(s.strip()) > 2]

    results = {aspect: [] for aspect in aspect_list}

    for sent in sentences:
        # 分割した1文をベクトル化
        sent_embedding = embedding_model.encode(sent, convert_to_tensor=True)

        # どの固定要素に最も意味が近いか類似度計算
        cosine_scores = util.cos_sim(sent_embedding, aspect_embeddings)[0]
        best_match_idx = torch.argmax(cosine_scores).item()
        best_score = cosine_scores[best_match_idx].item()

        # 類似度が一定以上（意味が関連している）場合のみ、その要素に割り当て
        if best_score > 0.3:
            matched_aspect = aspect_list[best_match_idx]
            pos_rate = predict_sentiment(sent)
            results[matched_aspect].append(pos_rate)

    # スコアの平均（%換算）
    final_scores = {}
    for aspect, score_list in results.items():
        if len(score_list) > 0:
            final_scores[aspect] = round(sum(score_list) / len(score_list), 2)
        else:
            final_scores[aspect] = None # 言及なし

    return final_scores

# ----------------------------------------------------
# テスト実行（手動辞書にない表現を含んだレビュー）
# ----------------------------------------------------
review = "お肉が柔らかくて最高でした！でも週末だったせいかかなり混雑していて、接客も雑に感じました。"

print("【分析結果（固定要素への自動分類）】")
scores = analyze_review_with_fixed_aspects(review, genre="飲食店")
for aspect, score in scores.items():
    if score is not None:
        print(f"- {aspect}: ポジティブ率 {int(score*100)}%")
    else:
        print(f"- {aspect}: 言及なし")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: cl-tohoku/bert-base-japanese-whole-word-masking
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

【分析結果（固定要素への自動分類）】
- 料理の味・品質: ポジティブ率 0%
- 接客・サービス: 言及なし
- 待ち時間・混雑度: ポジティブ率 84%
- 価格・コスパ: 言及なし


In [ ]:
# ----------------------------------------------------
# テスト実行（手動辞書にない表現を含んだレビュー）
# ----------------------------------------------------
review = "お肉が柔らかくて最高でした！でも週末だったせいかかなり混雑していて、接客も雑に感じました。"

print("【分析結果（固定要素への自動分類）】")
scores = analyze_review_with_fixed_aspects(review, genre="飲食店")
for aspect, score in scores.items():
    if score is not None:
        print(f"- {aspect}: ポジティブ率 {int(score*100)}%")
    else:
        print(f"- {aspect}: 言及なし")

In [ ]:
# ==========================================
# 0. 必要なライブラリのインストール
# ==========================================
!pip install transformers[torch] datasets accelerate fugashi ipadic mecab-python3 sentence-transformers gradio pandas torch -q

import re
import json
import torch
import numpy as np
import pandas as pd
import gradio as gr

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sentence_transformers import SentenceTransformer, util

# GPUが使えるか確認
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"使用デバイス: {device}")


# ==========================================
# 1. 擬似データセットの生成＆ファインチューニング
# (※本来は海外データセットを翻訳したdf_jaを読み込む想定です)
# ==========================================
print("\n--- 1. モデルのファインチューニング処理を開始 ---")

# 海外ホテル/レストランレビューから作成した翻訳データセットの疑似サンプル
# Label 1: Positive, Label 0: Negative
sample_data = [
    # Positive
    {"text": "部屋がとても広くてベッドも清潔、ぐっすり眠れました。", "label": 1},
    {"text": "料理の味がどれも絶品で、シェフのこだわりを感じました。", "label": 1},
    {"text": "スタッフの笑顔と親切な対応に感動しました。素晴らしいサービスです。", "label": 1},
    {"text": "混雑もなくスムーズに案内され、快適な時間を過ごせました。", "label": 1},
    {"text": "コスパが最高で、この値段でこの品質は期待以上です。", "label": 1},
    {"text": "お風呂が綺麗でアメニティも充実しており非常に満足です。", "label": 1},
    # Negative
    {"text": "部屋の掃除が行き届いておらず、髪の毛が落ちていて最悪でした。", "label": 0},
    {"text": "料理が冷めていて不味い。価格に見合わないクオリティ。", "label": 0},
    {"text": "フロントの対応がかなり雑で不快な思いをしました。", "label": 0},
    {"text": "1時間以上並ばされ、案内も遅くてうんざりしました。", "label": 0},
    {"text": "料金が高すぎる割に設備が古く、二度と利用しません。", "label": 0},
    {"text": "隣の音がうるさくて全く眠れませんでした。防音性が低いです。", "label": 0}
]

df_train = pd.DataFrame(sample_data)

# 日本語BERTベースモデル
BASE_MODEL_NAME = "cl-tohoku/bert-base-japanese-v3"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
ft_model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL_NAME, num_labels=2).to(device)

# データセット変換
raw_dataset = Dataset.from_pandas(df_train)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_dataset = raw_dataset.map(tokenize_function, batched=True)

# ファインチューニング設定
training_args = TrainingArguments(
    output_dir="./finetuned_sentiment_model",
    num_train_epochs=5,                    # デモ用にエポック数5で高速学習
    per_device_train_batch_size=4,
    logging_steps=2,
    save_strategy="no",
    use_cpu=(device == "cpu"),
    report_to="none"
)

trainer = Trainer(
    model=ft_model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

# ファインチューニング実行
print("ファインチューニングを実行中...")
trainer.train()
print("ファインチューニング完了！自前感情分析モデルがロードされました。")

# SentenceTransformer（意味類似度・要素分類用）の準備
embedding_model = SentenceTransformer('cl-tohoku/bert-base-japanese-whole-word-masking')


# ==========================================
# 2. 事前定義アスペクト（固定要素）の定義
# ==========================================
FIXED_ASPECTS = {
    "宿泊施設": {
        "部屋・清潔さ": "部屋やベッド、お風呂が綺麗で清潔だった。掃除が行き届いている。防音。",
        "接客・スタッフ": "フロントやスタッフの対応が親切丁寧で笑顔だった。案内がスムーズ。",
        "お風呂・設備": "温泉や大浴場、アメニティ、設備が充実していた。",
        "朝食・食事": "朝食バイキングや夕食の料理がとても美味しかった。"
    },
    "飲食店": {
        "料理の味・品質": "料理の味が美味しくて品質が高い。メニューが豊富。",
        "接客・サービス": "店員やスタッフの接客対応が良い。愛想が良い。",
        "待ち時間・混雑": "待ち時間が短くスムーズに入れた。混雑していない。",
        "価格・コスパ": "価格が安くコストパフォーマンスが高い。お得感がある。"
    },
    "観光地・レジャー": {
        "体験・見ごたえ": "景観やアトラクション、イベントが楽しく見ごたえがあった。",
        "混雑度・回りやすさ": "混雑しておらず快適に回れた。広い。",
        "スタッフ対応": "キャストやスタッフの案内が丁寧で親切だった。",
        "アクセス・設備": "駐車場やアクセスが良く、トイレや休憩所が綺麗だった。"
    }
}


# ==========================================
# 3. テキスト分解 & 感情判定ロジック
# ==========================================
def predict_sentiment_score(text):
    """自前ファインチューニングモデルでポジティブ確率(0.0〜1.0)を出力"""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(device)
    ft_model.eval()
    with torch.no_grad():
        outputs = ft_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0]
    return probs[1].item() # 1: Positive の確率

def split_sentences(text):
    """句点および逆説の接続詞（「〜だが」「〜けど」）で文章を細かく分解"""
    raw_sentences = [s.strip() for s in re.split(r'[。！!？?\n]', text) if len(s.strip()) > 1]
    fine_sentences = []
    for s in raw_sentences:
        sub_s = re.split(r'(?<=[だけど|ですが|ものの|けれど|が|のに])', s)
        fine_sentences.extend([ss.strip() for ss in sub_s if len(ss.strip()) > 1])
    return fine_sentences

def analyze_single_review(text, genre):
    """1件のレビュー文を分解し、固定要素への自動自動マッピング＋感情判定を実施"""
    aspect_defs = FIXED_ASPECTS.get(genre, FIXED_ASPECTS["宿泊施設"])
    aspect_names = list(aspect_defs.keys())
    aspect_anchor_texts = list(aspect_defs.values())

    # アンカーテキストをベクトル化
    anchor_embeddings = embedding_model.encode(aspect_anchor_texts, convert_to_tensor=True)

    sentences = split_sentences(text)
    results = {aspect: [] for aspect in aspect_names}

    for sent in sentences:
        sent_embedding = embedding_model.encode(sent, convert_to_tensor=True)
        # どの固定要素の定義に意味が近いか類似度計算
        cosine_scores = util.cos_sim(sent_embedding, anchor_embeddings)[0]
        best_idx = torch.argmax(cosine_scores).item()
        best_score = cosine_scores[best_idx].item()

        # コサイン類似度が一定以上（関連性が高い）場合のみ該当要素に割り当て
        if best_score > 0.35:
            matched_aspect = aspect_names[best_idx]
            pos_score = predict_sentiment_score(sent)
            results[matched_aspect].append(pos_score)

    # 各要素の平均ポジティブ率（%換算用）
    final_scores = {}
    for aspect, score_list in results.items():
        if len(score_list) > 0:
            final_scores[aspect] = round(sum(score_list) / len(score_list), 2)
        else:
            final_scores[aspect] = None # 言及なし

    return final_scores


# ==========================================
# 4. 複数レビューの集計 ＆ インサイト要約生成
# ==========================================
def process_multiple_reviews_pipeline(genre, raw_json_input):
    """API等から読み込んだ複数レビューデータを集計・要約"""
    try:
        reviews = json.loads(raw_json_input)
    except:
        return "⚠️ JSONフォーマットが不正です。正しいJSON構造を入力してください。", None

    results = []
    for r in reviews:
        text = r.get("text", "")
        scores = analyze_single_review(text, genre)
        results.append(scores)

    df = pd.DataFrame(results)

    target_aspects = list(FIXED_ASPECTS.get(genre, FIXED_ASPECTS["宿泊施設"]).keys())
    avg_scores = {}

    for col in target_aspects:
        if col in df.columns:
            valid_scores = df[col].dropna()
            if len(valid_scores) > 0:
                avg_scores[col] = round(float(valid_scores.mean()), 2)
            else:
                avg_scores[col] = 0.50 # 全く言及がない場合は中立50%
        else:
            avg_scores[col] = 0.50

    # 要約（AIインサイト）文の動的自動生成
    best_aspect = max(avg_scores, key=avg_scores.get)
    worst_aspect = min(avg_scores, key=avg_scores.get)

    best_pct = int(avg_scores[best_aspect] * 100)
    worst_pct = int(avg_scores[worst_aspect] * 100)

    summary_text = f"### 📊 【{genre}】 複数口コミ集計＆分析結果（対象件数: {len(reviews)}件）\n"
    summary_text += f"- 🌟 **強み (高評価要素):** 「**{best_aspect}**」（ポジティブ率 **{best_pct}%**）\n"

    if worst_pct < 50:
        summary_text += f"- ⚠️ **課題 (ネガティブ要因):** 「**{worst_aspect}**」（ポジティブ率 **{worst_pct}%**）に関する不満が全体の評価を押し下げています。\n"
    else:
        summary_text += f"- 💡 **改善の余地:** 全体的に好評価ですが、「**{worst_aspect}**」（ポジティブ率 **{worst_pct}%**）にさらなる向上の余地があります。\n"

    return summary_text, avg_scores


# ==========================================
# 5. Gradio Web UIの構築＆起動
# ==========================================
# テスト用APIレスポンス想定JSONデータ
sample_json_hotel = json.dumps([
    {"id": 1, "text": "部屋がすごく広くて綺麗でした！フロントのスタッフも親切。ただ、朝食バイキングの料理が冷めていて不味かったです。"},
    {"id": 2, "text": "大浴場と温泉は最高でした。でも、チェックインの待ち時間が1時間以上と長すぎてうんざりしました。"},
    {"id": 3, "text": "スタッフの対応が笑顔でとても好印象。ベッドもふかふかで満足です。コスパも良い！"},
    {"id": 4, "text": "隣の人の声が響いて部屋がうるさかった。価格の割にサービスも悪くてガッカリ。"}
], ensure_ascii=False, indent=2)

with gr.Blocks(title="口コミ要素別 感情集計・分析システム") as demo:
    gr.Markdown("# 🏢 口コミ要素別 満足度自動集計・要約システム")
    gr.Markdown("事前に指定した固定要素に対し、自前でファインチューニングした日本語モデルとベクトル類似度検索でポジ・ネガ率をパーセント算出します。")

    with gr.Row():
        with gr.Column():
            genre_dropdown = gr.Dropdown(
                choices=["宿泊施設", "飲食店", "観光地・レジャー"],
                value="宿泊施設",
                label="対象施設のジャンルを選択"
            )
            input_json = gr.Code(
                value=sample_json_hotel,
                language="json",
                label="複数レビューデータ (API受信用JSON)"
            )
            btn = gr.Button("複数レビューを一括分析・集計", variant="primary")

        with gr.Column():
            summary_output = gr.Markdown(label="全体分析要約レポート")
            chart_output = gr.Label(label="要素別 平均ポジティブ満足度 (%)")

    btn.click(
        fn=process_multiple_reviews_pipeline,
        inputs=[genre_dropdown, input_json],
        outputs=[summary_output, chart_output]
    )

print("\n--- 画面を起動します ---")
demo.launch(share=True, debug=True)

使用デバイス: cuda

--- 1. モデルのファインチューニング処理を開始 ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: cl-tohoku/bert-base-japanese-v3
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

ファインチューニングを実行中...


...

ファインチューニング完了！自前感情分析モデルがロードされました。


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: cl-tohoku/bert-base-japanese-whole-word-masking
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- 画面を起動します ---
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://dcb799c25b2884175c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://dcb799c25b2884175c.gradio.live


In [1]:
# ==========================================
# 0. 必要なライブラリのインストール
# ==========================================
!pip install transformers[torch] datasets accelerate fugashi ipadic mecab-python3 sentence-transformers gradio pandas torch -q

import re
import json
import torch
import numpy as np
import pandas as pd
import gradio as gr

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sentence_transformers import SentenceTransformer, util

#data(fine tuning)
# restaurants
splits = {'train': 'data/train-00000-of-00001.parquet', 'validation': 'data/validation-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
df_restaurants = pd.read_parquet("hf://datasets/cmotions/NL_restaurant_reviews/" + splits["train"])

print("レストランデータ:", df_restaurants.head())

#hotel
import kagglehub
from kagglehub import KaggleDatasetAdapter

# path パラメータで読み込むCSVファイルを明示的に指定する
df_hotel = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "jiashenliu/515k-hotel-reviews-data-in-europe",
    path="Hotel_Reviews.csv"  # <-- 対象のファイル名を指定
)

print("ホテルデータ:", df_hotel.head())

# GPUが使えるか確認
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"使用デバイス: {device}")

レストランデータ:    restaurant_ID restaurant_review_ID  michelin_label  score_total  \
0         258641            258641_24               0          5.8   
1         392579            392579_74               0          8.4   
2         251725             251725_4               0          8.4   
3         242541             242541_8               0          7.7   
4         243207           243207_113               0          8.0   

   score_food  score_service  score_decor fame_reviewer  reviewscore_food  \
0         6.5            6.0          6.2   Fijnproever                10   
1         8.9            8.5          8.7       Proever                 8   
2         8.9            8.7          8.7   Fijnproever                 8   
3         8.4            8.5          8.2       Proever                 2   
4         8.2            7.6          8.0   Fijnproever                 7   

   reviewscore_service  reviewscore_ambiance  reviewscore_waiting  \
0                    8               

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split

# ----------------------------------------------------
# 1. 前処理：ポジティブ文とネガティブ文を1つのデータフレームに結合
# ----------------------------------------------------
# df_hotel は読み込み済みのホテルデータとします
# 無意味なテキスト("No Negative", "No Positive")を除外
pos_df = df_hotel[df_hotel['Positive_Review'].str.strip() != 'No Positive'][['Positive_Review']].rename(columns={'Positive_Review': 'text'})
pos_df['label'] = 1  # ポジティブ = 1

neg_df = df_hotel[df_hotel['Negative_Review'].str.strip() != 'No Negative'][['Negative_Review']].rename(columns={'Negative_Review': 'text'})
neg_df['label'] = 0  # ネガティブ = 0

# サンプリング（学習時間を短正するため各5,000件ずつ抽出。本番は増やしてください）
dataset_df = pd.concat([pos_df.sample(5000, random_state=42), neg_df.sample(5000, random_state=42)]).reset_index(drop=True)

# 訓練データと検証データに分割
train_texts, val_texts, train_labels, val_labels = train_test_split(
    dataset_df['text'].tolist(), dataset_df['label'].tolist(), test_size=0.2, random_state=42
)

# ----------------------------------------------------
# 2. トークナイザーとモデルのロード
# ----------------------------------------------------
MODEL_NAME = "roberta-base" # 英語の高精度モデル
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# PyTorch用 Dataset クラス
class HotelReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len)
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = HotelReviewDataset(train_texts, train_labels, tokenizer)
val_dataset = HotelReviewDataset(val_texts, val_labels, tokenizer)

# ----------------------------------------------------
# 3. ファインチューニング（学習）の実行
# ----------------------------------------------------
training_args = TrainingArguments(
    output_dir='./results_hotel',
    num_train_epochs=2,              # エポック数
    per_device_train_batch_size=16,  # バッチサイズ
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    logging_steps=100,
    fp16=True,                       # T4 GPU利用時は高速化
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

# 学習スタート！
trainer.train()

# 学習済みモデルの保存
model.save_pretrained("./fine_tuned_hotel_model")
tokenizer.save_pretrained("./fine_tuned_hotel_model")
print("ホテル用モデルのファインチューニング完了！")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,0.163179,0.212696
2,0.104455,0.228108


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

ホテル用モデルのファインチューニング完了！


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split

# ----------------------------------------------------
# 1. 前処理：テキストと4つのアスペクトスコア（0〜1に正規化）を準備
# ----------------------------------------------------
# df_restaurant は読み込み済みのレストランデータとします
df_samle = df.restaurants(n=1000, random_state=42)
aspect_cols = ['reviewscore_food', 'reviewscore_service', 'reviewscore_ambiance', 'reviewscore_value']

# スコアを 0.0 〜 1.0 に正規化（元が10点満点のため10で割る）
df_res_clean = df_sample.dropna(subset=['review_text'] + aspect_cols).copy()
for col in aspect_cols:
    df_res_clean[col] = df_res_clean[col] / 10.0

train_df, val_df = train_test_split(df_res_clean, test_size=0.2, random_state=42)

# ----------------------------------------------------
# 2. 多出力回帰用のカスタムモデル定義
# ----------------------------------------------------
MODEL_NAME = "xlm-roberta-base" # 多言語（オランダ語対応）モデル
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class MultiAspectRegressionModel(nn.Module):
    def __init__(self, model_name, num_aspects=4):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.regressor = nn.Linear(self.encoder.config.hidden_size, num_aspects)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # [CLS] トークンのベクトルを取得して回帰ヘッダーへ
        cls_output = outputs.last_hidden_state[:, 0, :]
        logits = torch.sigmoid(self.regressor(cls_output)) # 0〜1の範囲で出力

        loss = None
        if labels is not None:
            loss_fct = nn.MSELoss() # 平均二乗誤差損失
            loss = loss_fct(logits, labels)

        return (loss, logits) if loss is not None else logits

# Dataset クラス
class RestaurantDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.encodings = tokenizer(df['review_text'].tolist(), truncation=True, padding=True, max_length=max_len)
        self.labels = df[aspect_cols].values.astype('float32')

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = RestaurantDataset(train_df, tokenizer)
val_dataset = RestaurantDataset(val_df, tokenizer)

# ----------------------------------------------------
# 3. ファインチューニング実行
# ----------------------------------------------------
multi_model = MultiAspectRegressionModel(MODEL_NAME, num_aspects=4)

training_args = TrainingArguments(
    output_dir='./results_restaurant',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    eval_strategy="epoch",
    learning_rate=3e-5,
    fp16=True,
)

trainer = Trainer(
    model=multi_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()

print("レストラン用マルチアスペクトモデルの学習完了！")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss


In [ ]:
# 推論テスト（日本語の口コミを入れてみる）
sample_text = "料理はとても美味しかったですが、店員さんの態度が冷たくて少し残念でした。"

inputs = tokenizer(sample_text, return_tensors="pt", truncation=True)
with torch.no_grad():
    predicted_scores = multi_model(inputs["input_ids"], inputs["attention_mask"])[0].tolist()

print(f"味スコア: {int(predicted_scores[0]*100)}%")
print(f"接客スコア: {int(predicted_scores[1]*100)}%")
print(f"雰囲気スコア: {int(predicted_scores[2]*100)}%")
print(f"コスパスコア: {int(predicted_scores[3]*100)}%")

In [ ]:
!pip install deep-translator -q
from deep_translator import GoogleTranslator

def predict_with_translation(text_jp):
    # 日本語から英語へ自動翻訳
    text_en = GoogleTranslator(source='ja', target='en').translate(text_jp)

    # 翻訳後の英語で推論
    inputs = tokenizer(text_en, return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0]

    pos_score = probs[1].item()
    print(f"【元文(日)】 {text_jp}")
    print(f"【翻訳(英)】 {text_en}")
    print(f" └ ポジティブ度: {int(pos_score * 100)}%\n")

# テスト実行
predict_with_translation("部屋は綺麗だったけど、フロントの対応がかなり悪かった。")

【元文(日)】 部屋は綺麗だったけど、フロントの対応がかなり悪かった。
【翻訳(英)】 The room was clean, but the reception at the front desk was very bad.
 └ ポジティブ度: 0%



In [ ]:
from huggingface_hub import hf_hub_download
f = hf_hub_download('jniimi/tripadvisor-review-rating', repo_type='dataset', filename='data1000.pkl')
df_hotel = pd.read_pickle(f)
print("ホテルデータ:", df_hotel.head())